# SPOD to Sharepoint integration

- Prerequisites: 
  - [Office365-REST-Python-Client](https://pypi.org/project/Office365-REST-Python-Client/) installed. `pip install Office365-REST-Python-Client`
  - Anaconda packages: `pandas, openpyxl`
- Access to a [list(s)](https://support.microsoft.com/en-us/office/introduction-to-lists-0a1c3ace-def0-44af-b225-cfa8d92c52d7) in Sharepoint / Office365.

Beware: Sharepoint has it's own way to encode special characters in names (List-Names, Column-Names, ...). See [Encoding](https://www.tachytelic.net/2021/06/encode-sharepoint-column-names-to-internal/)
Just use simple names as 'Entities' and all is fine.


## Structure

1. Define mapping between SPOD (json) and columns in the list
1. Ensure list columns conform mapping. Automatically update if need.
1. Scan current content and preserve it in a table (Pandas)
1. Define and show changeset that will be applied
1. Upload changes
1. Apply changes via execute_query()


## Configuration

In [ ]:
LIBRARY = '../../pythonWork/pythonSource'

# If None, the pip/conda installed package will be used
SHAREPOINT_LIBRARY_PATH = '../../lib/Office365-REST-Python-Client'

CONFIGURATION = 'fyayc-sika-preview.yaml' if 'CONFIGURATION' not in globals() else CONFIGURATION

apply_changes = True

## Check prerequisites

In [ ]:
import sys
import logging
import os
import json
import yaml
from pathlib import Path

In [ ]:
configfile = Path(CONFIGURATION)
assert configfile.is_file(), f"Cannot find configuration file '{configfile.resolve()}'"

with open(configfile, 'r') as src:
    configuration = yaml.safe_load(src)
assert configuration['sharepoint'] is not None
spconf = configuration['sharepoint']
print(f"Loaded configuration for sharepoint access with user '{spconf['credentials']['username']}' from {configfile}")

## Initialize logging

In [ ]:
import logging
from logging import handlers
from datetime import datetime

stamp = datetime.now()
run_stamp = stamp.strftime("%Y-%m-%d-%H-%M-%S")

os.makedirs('log', exist_ok=True)
logfile = f'log/sharepoint-list-sync-{run_stamp}.log'

handler = handlers.RotatingFileHandler(logfile, maxBytes=(1024 * 1024 * 10), backupCount=10)
handler.setLevel(logging.DEBUG)

formatter = logging.Formatter("%(asctime)s [%(threadName)s] - %(name)s - %(levelname)s - %(message)s")
handler.setFormatter(formatter)

console_log_handler = logging.StreamHandler()
console_formatter = logging.Formatter("%(levelname)s - %(message)s")
console_log_handler.setFormatter(console_formatter)
console_log_handler.setLevel(logging.INFO)

logger = logging.getLogger()
logger.setLevel(logging.DEBUG)
logger.addHandler(handler)
logger.addHandler(console_log_handler)

urlliblogger = logging.getLogger('urllib3.connectionpool')
urlliblogger.setLevel(logging.DEBUG)

In [ ]:
if SHAREPOINT_LIBRARY_PATH is not None:
    sp_lib_path = Path(SHAREPOINT_LIBRARY_PATH)
    assert sp_lib_path.is_dir(), f"{sp_lib_path.reslove()} is not a directory. The constant 'SHAREPOINT_LIBRARY_PATH' must point to the library. Default = None."
    sys.path.insert(0, str(sp_lib_path))

In [ ]:
try:
    from office365.sharepoint.lists.list import List
    from office365.runtime.auth.user_credential import UserCredential
    from office365.sharepoint.client_context import ClientContext
except:
    print("Office356 API library is missing. Install it with 'pip install Office365-REST-Python-Client'")
    raise

## Use the fyayc SPOD library

In [ ]:
toolpath = Path(LIBRARY)
assert toolpath.is_dir(), f"{toolpath.reslove()} is not a directory. The constant 'LIBRARY' must point to the library. Default = 'pythonWork/pythonSource'."
sys.path.insert(0, str(toolpath))

from PUBLISH_MODEL.sharepoint.list_publisher import update_structure, update_content, load_content

### Check SPOD

In [ ]:
spod_file = Path(configuration['spod'])
assert spod_file.is_file(), f"SPOD source missing: {spod_file.resolve()}"

In [ ]:
with open(spod_file, 'r') as src:
    spod = json.load(src)
print(f"Loaded {spod_file.resolve()}\n{spod['model']}\nVersion {spod['_imprint_']}")
mapdict = {}
for entry in spconf['lists']:
    key = next(iter(entry.keys()))
    print(f"- {key}: {len(spod[key])}")
    mapdict[key] = entry[key]['title']
print(f"Languages: {list(spod['languages'].keys())}")

## Access to Sharepoint site using Office365-REST-Python-Client library

In [ ]:
credentials = UserCredential(spconf['credentials']['username'], spconf['credentials']['password'])
ctx = ClientContext(spconf['site']).with_credentials(credentials)

### Verfiy connection / credentials

In [ ]:
from office365.runtime.http.request_options import RequestOptions 
request = RequestOptions("{0}/_api/web/".format(spconf['site']))
response = ctx.execute_request_direct(request)
response

In [ ]:
lists_available = ctx.lists.get().execute_query()
assert len(lists_available) > 0, f"Expecting more than 0 lists"
print(f"Found {len(lists_available)} lists in site {spconf['site']}")
print(f"Mapping exists for {len(mapdict)} tables: {list(mapdict.values())}")
for spl in lists_available:
    mapped = '✅' if spl.title in mapdict.values() else ''
    print(f"- {spl.title} {mapped} \t {spl.id}")

## Translation shortcut tr

In [ ]:
def tr(item, lang: str = 'en'):
    res = inner_tr(item, lang)
    if isinstance(res, dict):
        res = tr(res, lang)
    assert res is None or isinstance(res, str), f"Result is of type {type(res)}"
    return res

def inner_tr(item, lang) -> str:
    if isinstance(item, dict) and len(item) > 0:
        translation = item.get(lang)
        if translation is not None:
            return translation
        else:
            return next(iter(item.values()))
    if isinstance(item, str):
        return item
    
    if isinstance(item, list) and len(item) > 0:
        return tr(item[0])
    return ''

## List structure definition

In [ ]:
mappings = {}

### Defintion of the 'Entity' list

In [ ]:
import html
from urllib.parse import urlparse

def sharepoint_diagram_url(diagkey: str, lang: str) -> str:
    base = spconf['site'] + '/' + spconf['assets']['tools'] + '/Forms/AllItems.aspx?id='
    site_url = urlparse(spconf['site'])
    folder = '/' + spconf['assets']['svg-diagrams']
    target = site_url.path + folder + '/' + diagkey + '-' + lang + '.svg'
    sid = html.escape(target)
    parent = f"&parent={site_url.path}{folder}"
    return f"{base}{sid}{parent}"

def to_url_list(diagrams: list, lang: str) -> str:
    result = [ "<div>"]
    for diagkey in diagrams:
        title = spod['diagrams'][diagkey]['name']
        url = sharepoint_diagram_url(diagkey, lang)
        result.append(f"""<a href="{url}">{title}</a>""")
        result.append(", ")
    result = result[:-1]
    result.append("<div>")
    return ''.join(result)

In [ ]:
entity_mapping = [
    ('Title', {'value': lambda e: tr(e['name'], 'en'), 'properties': {
        'Description': 'Name of the entity',
        'FieldTypeKind': 2,
    }}),
    ('Description', {'value': lambda e: tr(e['descr'], 'en'), 'properties': {}}),
    ('Synonyms', {'value': lambda e: tr(e['synonyms'], 'en'), 'properties': {}}),
    ('Examples', {'value': lambda e: ', '.join(e['examples'].get('en', [])), 'properties': {}}),
    ('Diagrams', {'value': lambda e: to_url_list(e['diagrams+'], 'en'), 'properties': {
        'Description': 'Information Model Diagrams containing this entity',
        'FieldTypeKind': 3, 
        'FieldType': 'SP.FieldMultilineText',
        'RichText': True
    }
    }),
    
    ('Category', {'value': lambda e: spod['categories'][e['category']]['name'], 'properties': {}}),
    
    ('Titel', {'value': lambda e: tr(e['name'], 'de'), 'properties': {
        'Description': 'Name der Entität',
        'FieldTypeKind': 2,
    }}),
    ('Beschreibung', {'value': lambda e: tr(e['descr'], 'de'), 'properties': { 'Description': 'Beschreibung (de)' }}),
    ('Synonyme', {'value': lambda e: tr(e['synonyms'], 'de'), 'properties': { 'Description': 'Synonyme (de)' }}),
    ('Beispiele', {'value': lambda e: ', '.join(e['examples'].get('de', [])), 'properties': { 'Description': 'Beispiele (de)' }}),
    ('Diagramme', {'value': lambda e: to_url_list(e['diagrams+'], 'de'), 'properties': {
        'Description': 'Informationsmodell Diagramme die diese Entity zeigen',
        'FieldTypeKind': 3, 
        'FieldType': 'SP.FieldMultilineText',
        'RichText': True
    }}),
                    
    ('Titre', {'value': lambda e: tr(e['name'], 'fr'), 'properties': {
        'Description': 'Titre de l\'entité',
        'FieldTypeKind': 2,
    }}),
    ('Description', {'value': lambda e: tr(e['descr'], 'fr'), 'properties': { 'Description': 'Description (fr)' }}),
    ('Synonymes', {'value': lambda e: tr(e['synonyms'], 'fr'), 'properties': { 'Description': 'Synonymes (fr)' }}),
    ('Exemples', {'value': lambda e: ', '.join(e['examples'].get('fr', [])), 'properties': { 'Description': 'Exemples (fr)' }}),
    ('Diagrammes', {'value': lambda e: to_url_list(e['diagrams+'], 'fr'), 'properties': {
        'Description': 'Modèle d\'information Diagrammes montrant cette entité',
        'FieldTypeKind': 3, 
        'FieldType': 'SP.FieldMultilineText',
        'RichText': True
    }}),
    
    ('Status', {'value': lambda e: e['publstatus'], 'properties': {
        'Description': 'Publication status',
        'FieldTypeKind': 2,
        'FieldType': 'SP.FieldText',
        'Indexed': True,
        'Filterable': True,
        'Sortable': True,
    }}),
    ('Key', {'value': 'KEY', 'properties': {
        'Description': 'SSOT ID',
        'FieldTypeKind': 2,
        'FieldType': 'SP.FieldText',
        'Required': True,
        'Indexed': True,
        'AutoIndexed': True,
        'EnforceUniqueValues': True,
        'Filterable': True,
        'Sortable': True,
    }}),
    #   ('Documentation Link', {'value': lambda e: './bla.html', 'properties': {'FieldType': 11}}),
]
mappings['entities'] = entity_mapping

### Definition of the 'Attribute' list
This list contains **all** attributes of the IM

In [ ]:
attribute_mapping = [
    ('Title', {'value': lambda e: tr(e['name']), 'properties': {
        'FieldTypeKind': 2,
        'FieldType': 'SP.FieldText',
    }}),
    ('Description', {'value': lambda e: tr(e['descr']), 'properties': {
        'Description': 'Business description of the attribute',
        'FieldTypeKind': 3,
        'FieldType': 'SP.FieldMultilineText'
    }}),
    ('Type', {'value': lambda a: a['type+'], 'properties': {
        'Description': 'Datatype of the attribute',
        'Required': False,
        'FieldTypeKind': 2,
        'FieldType': 'SP.FieldText',
    }}),
    ('Key', {'value': 'KEY', 'properties': {
        'Description': 'SSOT ID',
        'FieldTypeKind': 2,
        'FieldType': 'SP.FieldText',
        'Required': True,
        'Indexed': True,
        'AutoIndexed': True,
        'EnforceUniqueValues': True,
        'Filterable': True,
        'Sortable': True,

    }}),
]
mappings['attributes'] = attribute_mapping

### Definition of the 'System' list

In [ ]:
systems_mapping = [
    ('Title', {'value': lambda e: tr(e['name']), 'properties': {
        'FieldTypeKind': 2,
        'FieldType': 'SP.FieldText',
    }}),
    ('Description', {'value': lambda e: tr(e['descr']), 'properties': {
        'Description': 'Description of the system',
        'FieldTypeKind': 3,
        'FieldType': 'SP.FieldMultilineText'
    }}),
    ('Key', {'value': 'KEY', 'properties': {
        'Description': 'SSOT ID',
        'FieldTypeKind': 2,
        'FieldType': 'SP.FieldText',
        'Required': True,
        'Indexed': True,
        'AutoIndexed': True,
        'EnforceUniqueValues': True,
        'Filterable': True,
        'Sortable': True,
    }}),
]
mappings['systems'] = systems_mapping

In [ ]:
try:
    sharepoint_list = ctx.lists.get_by_title('Entities')
    fields = sharepoint_list.fields.get().execute_query()
    the_field = next(iter(filter(lambda f: f.properties['EntityPropertyName'] == 'Diagrams', fields)))
    assert the_field
except Exception:
    import types
    the_field = types.SimpleNamespace()
    setattr(the_field, 'properties', {})
    pass

In [ ]:
the_field.properties

# Preparation steps

## Preparing list structure 

In [ ]:
ctx.clear()
apply_changes

In [ ]:
entitiy_list = None

for entry in spconf['lists']:
    key = next(iter(entry.keys()))
    mapping = mappings.get(key)
    if mapping is not None:
        sp_list = ctx.lists.get_by_title(entry[key]['title'])
        result = update_structure(sp_list, mapping, direct_write=apply_changes)
        print(f"List {sp_list.title} for {key}:\n{os.linesep.join(result)}")
        if 'entities' in key:
            entity_list = sp_list
        fields = sp_list.fields.get().execute_query()
        has_key = False
        for field in fields:
            name = field.properties['EntityPropertyName']
            if 'Key' == name:
                has_key = True
        assert has_key
    else:
        logging.warning(f"No mapping for {key}")

In [ ]:
ctx.execute_query()

In [ ]:
print(f"Found list with entities: {entity_list.title}")

# Synchronize content
1. Read content, store it to local backup
2. Apply changes if any

In [ ]:
from tqdm.autonotebook import tqdm

In [ ]:
spconf['lists']

In [ ]:
for entry in spconf['lists']:
    key = next(iter(entry.keys()))
    mapping = mappings.get(key)
    if mapping is not None:
        print(f"Processing list '{entry[key]['title']}'")
        sp_list = ctx.lists.get_by_title(entry[key]['title']).get().execute_query()
        
        content = load_content(sp_list.items)
        
        entries = spod[key]
        #entries = {}
        
        print(f"List '{entry[key]['title']}' currently contains {len(content)} entries. SPOD contains {len(entries)}")
        new, updated, deleted = update_content(sp_list, mapping, entries, content, direct_write=True)
        print(f" Creating: {len(new)}, updating: {len(updated)}, deleting: {len(deleted)} - executing {len(list(iter(ctx.pending_request())))} requests")
        sp_list.execute_query()
    else:
        logging.warning(f"No mapping for {key}")

In [ ]:
ctx.execute_query()

## Verify values written

In [ ]:
progressbar = None

def progress(items_read):
    progressbar.update(items_read)


for entry in spconf['lists']:
    key = next(iter(entry.keys()))
    mapping = mappings.get(key)
    if mapping is not None:
        sp_list = ctx.lists.get_by_title(entry[key]['title']).get().execute_query()
        items = sp_list.items
        count = sp_list.properties.get('ItemCount')
        with tqdm(total=count) as progressbar:
            items._page_size = 50
            items.page_loaded += progress

            print(f"Updating mapping of {entry[key]['title']} (SPOD <-> Sharepoint List) {count}")
            content = load_content(items)
            for spod_key, item in content.items():
                sp_id = item.properties['ID']
                spdict = spod[key][spod_key].get('sharepoint', {})
                spdict['list-item-id'] = sp_id
                spod[key][spod_key]['sharepoint'] = spdict
            progressbar.update(count)
            progressbar.close()

# Publish diagrams

In [ ]:
from office365.sharepoint.folders.folder import Folder

In [ ]:
destination = spconf['assets']['svg-diagrams'] 
print(f"Publishing {len(spod['diagrams'])} diagrams to {spconf['site']}/{destination}")

In [ ]:
folder = ctx.web.get_folder_by_server_relative_url(destination)
folder

In [ ]:
files = folder.files.execute_query().get()
files

In [ ]:
entity_list.properties

In [ ]:
assert entity_list is not None

def sharepoint_entity_links(svg: str, spod: dict) -> str:
    result = svg
    for key, entity in spod['entities'].items():
        sp_id = entity['sharepoint']['list-item-id']
        site_url = urlparse(spconf['site'])
        base = site_url.path + f"/Lists/{entity_list.title}/DispForm.aspx?ID={sp_id}&"
        turl = base
        
        spdict = entity.get('sharepoint', {})
        spdict['list-item-url'] = turl
        entity['sharepoint'] = spdict
        
        result = result.replace(f'href="#{key}"', f'href="{turl}" target="_self" rel="noopener"')
    return result

        base = spconf['site'] + f'/Lists/{entity_list.title}/DispForm.aspx?ID='
        site_url = urlparse(spconf['site'])
        folder = '/' + spconf['assets']['svg-diagrams']
        target = site_url.path + folder + '/' + diagkey + '.svg'
        sid = html.escape(target)
        parent = "&parent=" + folder
        title = spod['diagrams'][diagkey]['name']
        result.append(f"""<a href="{base}{sid}{parent}">{title}</a>""")

In [ ]:
# https://xyz.sharepoint.com/sites/xyz/Lists/Entities%20Integration/DispForm.aspx?ID=142&
sharepoint_entity_links('<a href="#ENTI7022">bla ENTI7022<a/>', spod)

In [ ]:
def load_svg_content(key: str, language: str) -> str:
    folder = Path(configuration['content'], 'svg-diagrams-' + language)
    assert folder.is_dir()
    hits = list(folder.glob("*" + key + ".svg"))
    assert len(hits) > 0, f"No asset found in '{folder}' matching *-{key}.svg"
    svg_src_file = hits[0]
    assert svg_src_file.is_file()

    with open(svg_src_file, 'r') as src:
        svg_content = src.read()

    processed = sharepoint_entity_links(svg_content, spod)
    
    # Write file for manual deployment or debugging purpose
    print(f"Rewrote size {len(svg_content)} -> {len(processed)}")
    outfile = Path(svg_src_file.parent, key + '-' + language +'.svg')
    with open(outfile, 'w') as out:
        out.write(processed)
    print(f"Wrote svg to {outfile}")
    
    return processed

In [ ]:
#processed = load_svg_content('DIAG701')

In [ ]:
for lang in spod['languages']:
    for key, diag in spod['diagrams'].items():
        processed = load_svg_content(key, lang)
        filename = key + '-' + lang + ".svg"
        files.upload(filename, processed.encode('utf-8'))
        ctx.execute_query()
        print(f"Uploaded {filename} to {files.resource_path}")
        diagdict = diag.get('sharepoint', {})
        diagdict['document'] = sharepoint_diagram_url(key, lang)
        diag['sharepoint'] = diagdict 

### Export as Excel for Upload

In [ ]:
from openpyxl import Workbook

def export_xlsx(sheet, mapping, content: dict) -> Workbook:
    column_index = 0
    header_row = sheet[0]
    for column in mapping:
        key = column[0]
        header_row[column_index] = key
        
        row_index = 1
        for key, value in content:
            sheet[row_index] = mapping
            
            options = column[1]
            map_function = options.get('value')
            try:
                value = None
                if type(map_function) is str:
                    if map_function == 'KEY':
                        value = key
                    else:
                        value = eval(map_function, {'e': entity})
                if callable(map_function):
                    value = map_function(entity)
                if value is not None:
                    values[tk] = value
            except Exception as e:
                print(f"{e}: {map_function}, mapping:{options}, tuple: {tuple}")
                logging.warning(
                    f"Cannot map field {tk} of entity {key}: {entity} to column {tk} with function {map_function}.\n",
                    e, exc_info=True)
            row_index += 1
            
        column_index += 1




In [ ]:
workbook = Workbook()
export_xlsx(workbook.active, entity_mapping, spod['entities'])

In [ ]:
for entry in spconf['lists']:
    key = next(iter(entry.keys()))
    mapping = mappings.get(key)
    if mapping is not None:
        workbook = export_xlsx(key, mapping)
        workbook.save(filename=f"{key}.xlsx")
    else:
        logging.warning(f"No mapping for {key}")

# Done